# Agentic AI, RAG & LLMs — Hands-On WorkshopWelcome! In the next two hours you will:1. Call a large language model from Python2. Learn how prompting changes what you get back3. Build a chatbot that remembers the conversation4. Build a **RAG** system from scratch (retrieval over your own documents)5. Build an **agent** that decides which Python function to call**You do not need to know Python well.** Every cell either runs as-is or has aclearly marked `TODO` for you to fill in.**How to run a cell:** click it, then press `Shift + Enter`.

---## Part 0 — Setup check (10 min)Run the next two cells. If you see a version number and then `Setup OK`, you are ready.

In [ ]:
# Installs everything we need. Takes about 60-90 seconds the first time.%pip install -q -U google-genai sentence-transformers numpyprint("Install finished.")

In [ ]:
from google import genaifrom google.genai import typesimport numpy as npMODEL = "gemini-2.5-flash"          # our main model for the whole workshopEMBED_MODEL = "gemini-embedding-001"  # used later, in the RAG sectionprint("Setup OK")

### Your API keyPaste the Gemini key you created at [aistudio.google.com](https://aistudio.google.com)into the cell below, between the quotes.> **Keep your own key.** Everyone must use their own — the free tier counts requests> per account, so sharing one key will rate-limit the whole room.

In [ ]:
API_KEY = "PASTE_YOUR_GEMINI_KEY_HERE"client = genai.Client(api_key=API_KEY)print("Client ready.")

---## Part 1 — Your first LLM call (10 min)An LLM takes text in and produces text out. That is the whole interface.- **Prompt** = the text you send- **Token** = roughly 3-4 characters; models read and write in tokens- **Model** = the specific trained system answering you (`gemini-2.5-flash` here)

In [ ]:
response = client.models.generate_content(    model=MODEL,    contents="Explain what a large language model is, in one sentence.")print(response.text)

In [ ]:
my_question = "Why is the sky blue? Answer in two sentences."response = client.models.generate_content(model=MODEL, contents=my_question)print(response.text)

**Notice:** run the same prompt twice and you may get different wording. LLMs areprobabilistic, not lookup tables. We control that next.

---## Part 2 — Prompting basics (20 min)Three levers you will use constantly:| Lever | What it does ||---|---|| **System instruction** | Standing orders — persona, rules, format || **Temperature** | 0.0 = focused and repeatable, 2.0 = wild and creative || **Few-shot examples** | Show 2-3 examples of the output you want |

In [ ]:
# A system instruction shapes every answer without cluttering your question.response = client.models.generate_content(    model=MODEL,    contents="What is a database?",    config=types.GenerateContentConfig(        system_instruction="You explain things to a curious 10-year-old. Two sentences maximum.",        temperature=0.2,    ),)print(response.text)

In [ ]:
response = client.models.generate_content(    model=MODEL,    contents="What is a database?",    config=types.GenerateContentConfig(        system_instruction="You are a pirate. Answer in pirate speak, two sentences maximum.",        temperature=1.5,    ),)print(response.text)

### Structured outputFree-form text is hard for programs to use. Ask for JSON and you can parse it.

In [ ]:
import jsonresponse = client.models.generate_content(    model=MODEL,    contents="List 3 fruits with their colours.",    config=types.GenerateContentConfig(        system_instruction="Reply with valid JSON only. No markdown, no backticks.",        temperature=0.0,    ),)data = json.loads(response.text)print(data)          # this is now a real Python objectprint(type(data))

In [ ]:
response = client.models.generate_content(    model=MODEL,    contents="List 4 European cities with their country and population, as a JSON array.",    config=types.GenerateContentConfig(        system_instruction="Reply with valid JSON only. No markdown, no backticks.",        temperature=0.0,    ),)data = json.loads(response.text)for city in data:    print(city["city"] if "city" in city else city)

---## Part 3 — Multi-turn chat (15 min)`generate_content` has **no memory** — each call starts from nothing.A chat session keeps the history and resends it every turn. That is all "memory" is.

In [ ]:
# Proof that a single call has no memory:print(client.models.generate_content(model=MODEL, contents="My name is Sam.").text)print("---")print(client.models.generate_content(model=MODEL, contents="What is my name?").text)

In [ ]:
# Now with a chat session, which carries the history:chat = client.chats.create(model=MODEL)print(chat.send_message("My name is Sam and I have 2 dogs.").text)print("---")print(chat.send_message("How many paws is that, and what is my name?").text)

In [ ]:
chat = client.chats.create(model=MODEL)print(chat.send_message("I'm planning a trip to Japan in April.").text)print("---")print(chat.send_message("What should I pack?").text)print("---")print(chat.send_message("Remind me which month I said.").text)for message in chat.get_history():    print(message.role, "->", message.parts[0].text[:80])

---## Part 4 — RAG from scratch (35 min)**The problem:** the model does not know your company handbook, your notes, oranything written after its training cut-off. Asked anyway, it may invent an answer.**RAG** (Retrieval-Augmented Generation) fixes this in three steps:1. **Embed** — turn each document into a list of numbers that captures its meaning2. **Retrieve** — find the documents whose numbers are closest to the question's numbers3. **Generate** — paste those documents into the prompt and ask the modelWe are building this ourselves, with numpy. No vector database, no framework.

### Step 1 — EmbeddingsAn embedding turns text into a vector (a list of numbers). Similar meanings landclose together. Run this to see what one actually looks like.

In [ ]:
from sentence_transformers import SentenceTransformer# Small, fast, free, runs locally on the CPU. First run downloads ~90 MB.embedder = SentenceTransformer("all-MiniLM-L6-v2")vector = embedder.encode("The cat sat on the mat.")print("Length of the vector:", len(vector))print("First 8 numbers:", vector[:8])

In [ ]:
# Similar meanings -> high similarity. Different meanings -> low.def similarity(a, b):    va, vb = embedder.encode([a, b], normalize_embeddings=True)    return float(va @ vb)   # dot product of normalised vectors = cosine similarityprint("dog/puppy      ", round(similarity("I love dogs", "Puppies are wonderful"), 3))print("dog/tax return ", round(similarity("I love dogs", "Please file your tax return"), 3))

### Step 2 — RetrievalHere is our tiny "knowledge base". In a real system these would be chunks of yourown PDFs or wiki pages.

In [ ]:
documents = [    "The Eiffel Tower is in Paris and was completed in 1889.",    "Python is a programming language created by Guido van Rossum in 1991.",    "The Great Wall of China is over 21,000 kilometres long.",    "Our office wifi password is 'workshop2026' and the guest network is 'GuestNet'.",    "The company holiday policy allows 25 days of paid leave per year.",    "Coffee is made from roasted coffee beans grown near the equator.",]# Embed every document once, up front. normalize_embeddings=True lets us use a# plain dot product for cosine similarity.doc_vectors = embedder.encode(documents, normalize_embeddings=True)print("Shape:", doc_vectors.shape, "-> 6 documents, 384 numbers each")

In [ ]:
def retrieve(question, k=2):    """Return the k documents most similar in meaning to the question."""    q_vector = embedder.encode([question], normalize_embeddings=True)[0]    scores = doc_vectors @ q_vector    top_indexes = np.argsort(scores)[::-1][:k]    return [documents[i] for i in top_indexes]print(retrieve("What is the wifi password?"))

Try `retrieve("how much time off do I get?")`. Notice it finds the holiday policyeven though the question shares almost no words with it. That is the point ofembeddings — they match **meaning**, not keywords.

In [ ]:
print(retrieve("how much time off do I get?"))print(retrieve("tell me about tall buildings in France"))

### Step 3 — GenerateFirst, see the model fail without retrieval. Then give it the context.

In [ ]:
question = "What is the office wifi password?"# WITHOUT retrieval — the model has no way to know this.print(client.models.generate_content(model=MODEL, contents=question).text)

In [ ]:
def ask_with_rag(question, k=2):    context = "\n".join(retrieve(question, k))    prompt = f"Context:\n{context}\n\nQuestion: {question}"    response = client.models.generate_content(        model=MODEL,        contents=prompt,        config=types.GenerateContentConfig(            system_instruction=(                "Answer using ONLY the provided context. "                "If the answer is not in the context, say you don't know."            ),            temperature=0.0,        ),    )    return response.textprint(ask_with_rag("What is the office wifi password?"))

In [ ]:
# The important test: does it admit when it doesn't know?print(ask_with_rag("How many days of leave do I get?"))print("---")print(ask_with_rag("Who won the 2026 World Cup?"))

In [ ]:
documents.append("The workshop instructor's favourite programming language is Python.")documents.append("This workshop runs for two hours and covers LLMs, RAG and agents.")doc_vectors = embedder.encode(documents, normalize_embeddings=True)print(ask_with_rag("How long is this workshop and what does it cover?"))

**That is RAG.** Roughly 15 lines. Production systems add chunking, betterembedding models, a vector database (Chroma, FAISS, Qdrant) and re-ranking — but thethree steps never change.

---## Part 5 — Your first agent (25 min)So far the model only produces text. An **agent** can *act*: you hand it somePython functions ("tools"), and it decides on its own which to call, with whicharguments, and what to do with the result.The loop is: **think → choose a tool → run it → read the result → answer.**Two things make a good tool: a clear **docstring** and **type hints**. The SDK readsboth to tell the model what the tool does.

In [ ]:
# LLMs are famously unreliable at arithmetic. Watch:print(client.models.generate_content(    model=MODEL,    contents="What is 4738 * 2913? Reply with just the number.").text)print("The real answer is:", 4738 * 2913)

In [ ]:
def multiply(a: float, b: float) -> float:    """Multiply two numbers together and return the result."""    print(f"   [tool called: multiply({a}, {b})]")    return a * bresponse = client.models.generate_content(    model=MODEL,    contents="What is 4738 * 2913?",    config=types.GenerateContentConfig(tools=[multiply]),)print(response.text)

You should see the `[tool called: ...]` line print. The model did not do the maths —it recognised it needed a tool, called your Python function, and used the answer.Now let's give it more than one tool, including our RAG retriever.

In [ ]:
def lookup_knowledge_base(query: str) -> str:    """Look up facts about the office, company policy and general trivia."""    print(f"   [tool called: lookup_knowledge_base('{query}')]")    return " ".join(retrieve(query))def word_count(text: str) -> int:    """Count how many words are in a piece of text."""    print(f"   [tool called: word_count(...)]")    return len(text.split())response = client.models.generate_content(    model=MODEL,    contents=(        "How many holiday days do we get, and what is 25 multiplied by 8 "        "(the number of hours that adds up to)?"    ),    config=types.GenerateContentConfig(        tools=[multiply, lookup_knowledge_base, word_count]    ),)print(response.text)

Watch which tools printed. The model picked them itself — you never wrote an `if`statement deciding when to search versus when to calculate. **That is the agentic part.**

In [ ]:
from datetime import datedef days_until(target_date: str) -> int:    """Return the number of days from today until the given date (format YYYY-MM-DD)."""    print(f"   [tool called: days_until({target_date})]")    y, m, d = map(int, target_date.split("-"))    return (date(y, m, d) - date.today()).daysresponse = client.models.generate_content(    model=MODEL,    contents="How many days until 2027-01-01?",    config=types.GenerateContentConfig(tools=[days_until]),)print(response.text)

---## Part 6 — Where to go next (10 min)You have now built, from scratch, the three things every LLM product is made of.**What you'd add for production**- *Chunking* — split long documents into ~500-token pieces before embedding- *A vector database* — `chromadb` is the easiest next step; FAISS or Qdrant at scale- *Better embeddings* — BGE, GTE or `gemini-embedding-001` beat MiniLM on accuracy- *Evaluation* — measure whether retrieved documents were actually relevant- *Guardrails* — the "answer only from context" instruction is your first defence  against hallucination, not your last**Frameworks worth learning next**| Tool | Good for ||---|---|| **LangChain / LangGraph** | Wiring multi-step pipelines and stateful agents || **LlamaIndex** | RAG-focused: loaders, chunking, indexes || **smolagents** (Hugging Face) | Minimal agents that write and run code || **PydanticAI** | Type-safe agents with very little boilerplate |**Free courses**- Hugging Face Agents Course — `huggingface.co/learn/agents-course` (free, Apache-2.0)- Google AI Studio docs — `ai.google.dev/gemini-api/docs`**One safety note:** free-tier prompts may be used to improve the provider's models.Never paste customer data, credentials or anything confidential into a free-tier API.---### Appendix — Backup provider (only if Gemini is rate-limited)If you hit a `429 RESOURCE_EXHAUSTED` error, switch to Groq. Same lesson, one changed cell.

In [ ]:
# Only run this if Gemini is unavailable.# %pip install -q openai## from openai import OpenAI# groq = OpenAI(api_key="YOUR_GROQ_KEY", base_url="https://api.groq.com/openai/v1")# reply = groq.chat.completions.create(#     model="llama-3.3-70b-versatile",#     messages=[{"role": "user", "content": "Hello!"}],# )# print(reply.choices[0].message.content)